# 03. Occupation numbers and the pairing iteration

This notebook explains how each diagonalization produces occupations and a proposed pairing field, and how the next iteration uses those results.

Read each explanation together with the code directly below it. The code is original Mathematica; the `.nb` files are the executable versions. Diagrams are existing saved results. These reading notebooks do not rerun the calculation.

## 1. Set the Fermi function and starting values

`f[x]` gives the occupation weight for an energy `x`. The second definition replaces the first: energies above `0.02` receive weight 0, energies below `-0.02` receive weight 1, and intermediate energies use `1/(Exp[Re[x]/kT]+1)`. `Re[x]` selects the real part. `kT` sets the energy scale over which the weights change.

`vs` is the pairing coupling, `mu` is the initial chemical-potential parameter, and `nexp` is the target mean occupation per site. The two files use different values:

| File | vs | nexp | initial mu | initial ds |
| --- | ---: | ---: | ---: | ---: |
| BdG topological.nb | -1.45 | 1.2 | 0 | -0.01 |
| added vortex.nb | -1.25 | 1.0 | 0 | -0.01 |

`st=Table[ds,...]` creates one entry per site. In a fresh session, `ds` is still symbolic at this statement and receives its value on the next line. Running the cells in their original order matters. The large section enclosed in `(* ... *)` is inactive.

**Occupation function and initialization** — `BdG topological.nb`, input 3.

```mathematica


f[x_]:=1./(Exp[x/kT]+1.);
kT = 0.001 (*eV*);
f[x_] :=(*UnitStep[x]*)(*1./(Exp[x/kT]+1.);*)If[Re[x] > 0.02, 0.0,
   If[Re[x] < -0.02, 1.0,
    1./(Exp[Re[x]/kT] + 1.)]];

(*mtuss={};*)
(*For[k=0,k\[LessEqual]0.2,k=k+0.1,
Print[StringJoin["\<calculte..\>",ToString[k]]];*)
(*flag1 = 1;*)
(*m = 0.5*0;*)
(*Us = 0.0;*)
vs = -1.45;(*s-wave pairing interaction*)
(*vd = -0.2*0; d-wave pairing interaction*)
mu = 0;

nexp = 1.2; (*expected electro density in your system*)
(*ntot = 1;
nut=Table[(ntot/2.)+((m*(-1)^(xpos[i]+ypos[i]))/2),{i,1,NN}];
ndt=Table[(ntot/2.)-((m*(-1)^(xpos[i]+ypos[i]))/2),{i,1,NN}];*)

Clear[v,as,b,bm];

(*dsignn={}; (*initialisation of d-wave sign*)
For[i=1,i<=NN,i++,jisel={};
For[j=1,j<=NN,j++,
If[xpos[i]==xpos[j],
If[ypos[i]==ypos[j]+1,AppendTo[jisel,-1],
If[ypos[i]==ypos[j]-1,AppendTo[jisel,-1],
If[ypos[i]==ypos[j]+(Ny-1),AppendTo[jisel,-1],
If[ypos[i]==ypos[j]-(Ny-1),AppendTo[jisel,-1]]]]],
If[ypos[i]==ypos[j],
If[xpos[i]==xpos[j]+1,AppendTo[jisel,1],
If[xpos[i]==xpos[j]-1,AppendTo[jisel,1],
If[xpos[i]==xpos[j]+(Nx-1),AppendTo[jisel,1],
If[xpos[i]==xpos[j]-(Nx-1),AppendTo[jisel,1]]]]]]]];
AppendTo[dsignn,jisel]];
spmtnn=Table[dsignn[[i,j]]*dijnn,{i,1,NN},{j,1,4}];
dijnn=0.3;*)

st=Table[ds,{i,1,NN},{j,1}]; (*initialisation of s-wave*)
ds=-0.01; (*Gap size*)
(*AbsoluteTiming[*)
(*MatrixForm[SCs[st]]*)
```

## 2. Diagonalize using the current fields

`iter=100` fixes the number of passes through the loop. Each pass first constructs `H` using the current `mu` and `st`, then evaluates `Eigensystem[H]`. The program does not stop early when the fields change by a small amount. The next excerpt shows the beginning of that loop; its closing bracket appears in the update section below.

**Beginning of the iteration** — `BdG topological.nb`, input 5.

```mathematica
(*calculation mean fields via iteration*)(*Main Hamiltonian BDG*)
iter = 100;
  (*em =  SparseArray[{x_, x_} -> 0, {NN, NN}];*)
For[a = 1, a <= iter, a++,
  (*mtus={};*)
  Huu = H1-Chempot1[mu] (*+imp*)(**+Hartree*);
  Hdd = H1-Chempot1[mu];
Hud = soc;
Hdu = Conjugate[soc];
hh1 = ArrayFlatten[{{Huu,Hud}, {Hdu, Hdd}}];
hh2 = ArrayFlatten[{{-Conjugate[Huu],Conjugate[Hud]}, {Conjugate[Hdu], -Conjugate[Hdd]}}];
    Huv= SCs[st]; (*Superconducting part \[CapitalDelta]*)
  Hvu=Conjugate[Huv];
  H = ArrayFlatten[{{hh1, Huv}, {Hvu, hh2}}]; (*Joins all the matrices*)
 (* Print[HermitianMatrixQ[H]];*)
  (*Print[MatrixForm[H]];*)
  
  ES = Eigensystem[H];
 

```

## 3. Calculate occupation and spin-population differences

`Map[f, ES[[1]]]` creates one occupation weight for every eigenvalue. `Abs[component]^2` gives a nonnegative component weight, and the dot product sums that weight over all eigenstates.

`nutcalc` uses the first component group and `fermi`. `ndtcalc` uses the fourth group and `1-fermi`. `ni` adds the two contributions; `mi` subtracts them. `nt` is the mean occupation per site. `mt` averages `Abs[mi]`, so opposite signs at different sites do not cancel. These sums use the entire returned eigensystem.

**Occupations and averages** — `BdG topological.nb`, input 5.

```mathematica
  fermi = Map[f, ES[[1]]];
  (*tab = Table[1,{i,Length[fermi]}];*)
  nutcalc = Table[((Abs[ES[[2]][[;; , i]]])^2.) . fermi, {i, 1, NN}];
  
  
  ndtcalc = Table[((Abs[ES[[2]][[;; , i + 3NN]]])^2.) . (1. - fermi), {i, 1, NN}];
  
  
  ni = nutcalc + ndtcalc;
  mi = nutcalc - ndtcalc;
  nt = Total[ni]/NN;
  mt = Total[Abs[mi]]/NN;
  
  
  
  (*Print["\<till here\>"];*)

```

## 4. Form the proposed pairing field

For each site, the code multiplies components from different groups and complex-conjugates the second factor. It adds the two products, weights them by `fermi`, sums over states, and multiplies by `vs`.

`Inner[Times,...,List]` creates the element-by-element products. The following dot product with `fermi` performs their weighted sum. Since `js[[i]]` contains the same site, this produces one onsite proposal per site in `stcalc`.

**Pairing proposal** — `BdG topological.nb`, input 5.

```mathematica
  stcalc=Table[vs*((Inner[Times,ES[[2]][[;;,i]],Conjugate[ES[[2]][[;;,j+
  3NN]]],List])+(Inner[Times,ES[[2]][[;;,i+NN]],Conjugate[ES[[2]][[;;,i+
  2NN]]],List])).fermi,{i,1,NN},{j,js[[i]]}];

```

## 5. Print progress and update the fields

`sta` is the arithmetic mean of the complex entries of `stcalc`. The printed `Abs[sta]` is the magnitude of that mean; it is not the mean of the individual magnitudes.

The chemical-potential update is `mu = mu - 0.1 (nexp-nt)`. If `nt` is below the target, this lowers `mu`; if it is above the target, it raises `mu`. In this code the electron diagonal contains `+mu`, which explains this feedback direction.

The pairing update `st = 0.5 stcalc + 0.5 st` keeps half the previous field and half the proposal. This mixing moderates changes between iterations. A fixed count of 100 updates does not demonstrate convergence by itself.

**Progress output, updates and exports** — `BdG topological.nb`, input 5.

```mathematica
  sta=Total[Flatten[stcalc]]/NN;(*
  stann=Total[Total[Abs[spmtcalcnn]]]/NN*0.25;*)
  
  
  
  Print[a , "\< mu=\>", mu, "\<  mt=\>", mt,(*"\< Us=\>", Us,*)"\<  n=\>", nt ,"\<  sup=\>", Abs[ sta(*"\<  supnn=\>",  stann*)]];
  mu = mu - 0.1*(nexp - nt);
  (*nut=0.5*nutcalc+0.5*nut;
  ndt=0.5*ndtcalc+0.5*ndt;*)
  st=0.5*stcalc+0.5*st;
  (*spmtnn=0.5*spmtcalcnn+0.5*spmtnn;*)
  ];
(*AppendTo[mtuss,{vd,stann}];
];*)


Export["\<Huu.wdx\>",Huu];
Export["\<Hud.wdx\>",Hud];
Export["\<Hdu.wdx\>",Hdu];
Export["\<Hdd.wdx\>",Hdd];
Export["\<SCs.wdx\>",SCs[st]];
```

## 6. Understand the exported arrays

`Huu.wdx`, `Hud.wdx`, `Hdu.wdx` and `Hdd.wdx` store the spin blocks left from the last matrix construction. `SCs.wdx` stores `SCs[st]` after the final pairing update. Because the last update follows the last diagonalization, the saved pairing field and saved normal blocks are not all recomputed together in a final extra pass.

The `Print` line reports the iteration index, current `mu`, `mt`, `nt`, and `Abs[sta]`. These values are progress information. There is no active convergence-curve plotting command in these cells, so this notebook does not add a convergence diagram.